In [19]:
import torch
from torch import Tensor


from botorch_community.models.np_regression import NeuralProcessModel
from botorch_community.acquisition.latent_information_gain import LatentInformationGain
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.likelihoods import GaussianLikelihood
from botorch.fit import fit_gpytorch_mll_torch
from botorch.optim import optimize_acqf
from botorch.test_functions import Hartmann
from botorch.utils.transforms import unnormalize

# Model fitting - what's the intended model fitting procedure?

In [20]:
DIM = 3
fun = Hartmann(dim=DIM)
train_X = torch.rand(5, DIM)
train_Y = fun(train_X)
model = NeuralProcessModel(train_X, train_Y)


#RuntimeError: All MarginalLogLikelihood objects must be given a GP object as a model. 
# If you are using a more complicated model involving a GP, pass the underlying GP object as the model, not a full PyTorch module.
mll = ExactMarginalLogLikelihood(model.likelihood, model)

# doesn't get here
fit_gpytorch_mll_torch(mll)

RuntimeError: All MarginalLogLikelihood objects must be given a GP object as a model. If you are using a more complicated model involving a GP, pass the underlying GP object as the model, not a full PyTorch module.

# Acquisition function optimization - Shapes need fixing - see below for explanation

In [23]:
# Acquisition function should take shape N x q x D and output shape N
# The unit tests currently do not do that. The q-dim should not be 
# brought in by unsqueezing, but is a dimension that ensures that multiple
# points can be computed in batch. Please look at other information-theoretic
# acquisition functions to see how this is done. The KL divergence can be
# computed for many points jointly, so this is probably what you want.

acq = LatentInformationGain(model)
candidate = optimize_acqf(
    acq,
    q=3,
    raw_samples=512,
    num_restarts=4,
    bounds=fun.bounds
)

RuntimeError: Tensors must have same number of dimensions: got 2 and 1